# Generate amorphous LSU networks

This notebook demonstrates `generate_lsu_network`, which implements the
Wooten-Winer-Weaire simulated annealing algorithm of Sellers et al.
(*Nat. Commun.* **8**, 14439, 2017) on a periodic 3-regular graph.

Documentation: see the `claude_context/` folder.

**Inputs**: target LSU (`lsu_degree_12` or `lsu_degree_22`), one of
(`num_vertices` or `num_rods`), `bounds_microns`.

**Output**: NumPy array `(R, 6)` where each row is `[x1, y1, z1, x2, y2, z2]` —
directly usable in the `create_permittivity_grid_penlike` pipeline. With the
default `pbc_duplicate_boundary_rods=True`, `R` equals the unique-edge count
plus the number of edges crossing box faces (each rendered twice, once from
each canonical-box endpoint), matching the Sellers reference file convention.

In [ ]:
import numpy as np
import lsu_network as lsu
import jax 
from datetime import datetime
import os
import platform

today = datetime.today()
formatted = today.strftime("%Y%m%d")

print('JAX available:', lsu.HAS_JAX)
print("Devices:", jax.devices())

def shutdown(delay_minutes=1):
    os.system(f"sudo shutdown -h +{delay_minutes}")

## Reproduce the example: 1000 vertices / 1500 unique edges, periodicity 11.44 µm

The reference example (`Example/lsu_example_ends.txt`) has Φ_12 ≈ 0.98 and
Φ_22 ≈ 0.89 (Sellers Eq. 2 convention: Φ_nl = depth-n trees, root vertices
within l edges; measured 0.9849 / 0.8887 with this repo's `compute_lsu`),
**N=1000 vertices, E=1500 unique edges**, periodicity 11.44 µm,
mean rod length 0.8 µm. The 1653 lines in that file include 153 PBC-image
duplicates of edges crossing box faces — required so that
`create_permittivity_grid_penlike` (which draws each rod as a literal cylinder
and does not apply PBC) produces a periodic permittivity grid.

At full scale this needs ~50,000 WWW iterations. With JAX it's tractable
(JIT-compiled energy + autodiff gradient); without JAX, plan to run
overnight or reduce iterations.

In [ ]:
# Production run. With num_vertices=1000 and the Sellers-confirmed energy
# weights this targets the Sellers reference example (Phi_22=0.89,
# Phi_12=0.98, L=11.44, d0=0.8). The pipeline is:
#
#   1. Seed network (crystal_srs or random_bm2000).
#   2. Initial full-N L-BFGS pulls bonds to d0.
#   3. Triangular WWW burn-in (Hemmann 2026 Sec. 2.3, Fig. 2): heat 0->T_max,
#      cool T_max->0, quench at T=0. Auto-calibrates T_max against T_melt
#      via Hemmann Eq. 5 (P_melt=0.001): T_melt = dE_min / ln(1/P_melt),
#      where dE_min is the relaxed cost of the energetically LOWEST uphill
#      bond switch of the seed (probed on 600 candidate switches from the
#      fixed initial state).
#   4. Production WWW with the same Vink/MB threshold-energy relax until
#      target Phi_22 reached or n_www_iterations exhausted.
#   5. Final clean-up relax.
#
# Notes on the kwargs (Sellers supplement refs [13]=Vink 2001 PRB 64 /
# [14]=Mousseau-Barkema 2001; BM2000 = PRB 62, 4985):
#
#   - seed_kind='crystal_srs' is the gyroid (srs) parent (Hemmann Z=3 recipe).
#     Switch to 'random_bm2000' for Sellers's literally-cited random seed
#     (Poisson placement + Hamiltonian cycle + chord matching loop expansion).
#   - threshold_energy_relax=True wires the BM2000 Eq. 3/4 early-rejection
#     scheme into every SW move: E_t = E_b - T*ln(s); abort the relax when
#     E - c_f|F|^2 > E_t, with rejections active only for relax cycles 6..10
#     (BM2000: none during the first 5 cycles; Hemmann Sec. 2.1: none after
#     cycle 10). Vink local->global rescue at cycle 10 if E sits within 0.1
#     ABOVE E_t. Same Metropolis identity as standard WWW (the final roll
#     reuses the same s), ~5-10x speedup on rejected moves.
#   - burn_in_T_max_over_T_melt: T_melt is now the faithful Hemmann Eq. 5
#     value (lowest-switch dE_min), which comes out ~2x SMALLER than the
#     pre-2026-06-11 mean-uphill estimate. The factor 5.0 below reproduces
#     the previously tuned effective burn-in temperature (old 2.5 x the
#     mean-based T_melt). Hemmann's hyperuniform window is 1.0-1.3 x T_melt
#     (Fig. 8) if you want their regime instead; note the production
#     initial_temperature=0.55 is ~10 x T_melt and will melt the network
#     regardless.
#   - uniformity_weight adds a low-k structure-factor penalty to Metropolis
#     acceptance only (not to L-BFGS), per Hemmann's note that local bonded
#     strain energy does not control long-wavelength density fluctuations.
#     Set 0.0 for strict Sellers Eq. 2 acceptance.
#   - energy_weights alpha=0.7, beta=0.7, gamma=0.3, delta=0.4 are the exact
#     values confirmed by the Sellers group for the Eq. 2 functional (also
#     the library defaults). Do not change them.

rods = lsu.generate_lsu_network(
    lsu_degree_22=0.89,                 # Type-2 amorphous gyroid target
    num_vertices=1000,                  # matches Example/lsu_example_ends.txt
    bounds_microns=11.44,
    edge_length=0.8,

    # --- Seed ---
    seed_kind='crystal_srs',            # switch to 'random_bm2000' to compare
    seed_lattice='srs',
    seed_jitter_sigma=0.10,

    # --- Triangular topology burn-in (Hemmann 2026 Sec. 2.3) ---
    burn_in_n_heat=6_000,
    burn_in_n_cool=12_000,
    burn_in_n_quench=4_000,
    burn_in_T_max=None,                  # auto via Hemmann Eq.5 (P_melt=0.001)
    burn_in_T_max_over_T_melt=5.0,       # = old effective 2.5x mean-based T_melt; see note above
    burn_in_P_melt=0.001,
    burn_in_T_melt_probe_moves=600,
    burn_in_T_melt_probe_T=5.0,          # unused since the Eq. 5 fix (kept for back-compat)
    burn_in_target_accepts_per_vertex=None,  # let schedule control length

    # --- Production WWW ---
    n_www_iterations=100_000,
    initial_temperature=0.55,          # calibrated to land near Sellers's Phi_22=0.89
    final_temperature=1e-3,
    target_tolerance=0.0001,
    check_lsu_every=500,

    # --- Relaxation (Vink/MB threshold-energy early rejection) ---
    relax_local_iters=120,
    relax_global_iters=300,
    threshold_energy_relax=True,
    c_f=0.3,                      # BM2000 Eq. 4 estimator constant (conservative, < 1)
    cycle_size=None,                     # auto ~25 cycles per move (Hemmann Sec. 2.1)
    global_fallback_threshold=float('inf'),  # disabled; in-relax promotion does the job
    local_shell_depth=4,                 # Vink 4th-neighbour moving shell
    # --- Acceptance objective (low-k uniformity penalty) ---
    uniformity_weight=0.5,
    uniformity_kmax=2,

    # --- Energy weights (Sellers-group confirmed; also the defaults) ---
    energy_weights={'alpha': 0.7, 'beta': 0.7, 'gamma': 0.3, 'delta': 0.4},

    seed=42,
    use_jax=True,
    verbose=True,
)
print('shape:', rods.shape)

## Save the output

The 6-column form (x1,y1,z1,x2,y2,z2) is directly compatible with
`np.loadtxt` as used by the rest of the pipeline. The 7-column form below
(with a 1-based index column) matches `Example/lsu_example_ends.txt`.

In [ ]:
os.makedirs('./Example', exist_ok=True)
filename = f'./Example/{formatted}_lsu_generated.txt'
# 6-column compatible with create_permittivity_grid_penlike
np.savetxt(filename, rods,
           fmt=' '.join(['%.14g'] * 6), delimiter='\t')

# # 7-column with index, matching Example/lsu_example_ends.txt
# indexed = np.column_stack([np.arange(1, len(rods) + 1), rods])
# np.savetxt(f'./Example/{formatted}_lsu_generated_indexed.txt', indexed,
#            fmt='%d\t' + '\t'.join(['%.14g'] * 6))

print('saved', rods.shape[0], 'rods')

## Verify the result

Quick checks: connectivity (rods belong to one connected network), edge length
distribution, and final LSU values.

In [ ]:
BOX = 11.44  # Must match bounds_microns above for the stats below to be meaningful.
p1 = rods[:, :3]
p2 = rods[:, 3:]
lengths = np.linalg.norm(p2 - p1, axis=1)

# 1) Rod-length distribution. Reference example (1653 rods, BOX=11.44):
#    mean=0.800 std=0.029  q5=0.752 med=0.801 q95=0.846 min=0.667 max=0.884
qs = np.quantile(lengths, [0.0, 0.05, 0.25, 0.5, 0.75, 0.95, 1.0])
print(f'rod count : {len(rods)}')
print(f'lengths   : mean={lengths.mean():.3f} std={lengths.std():.3f}')
print(f'  quartiles  min={qs[0]:.3f}  5%={qs[1]:.3f}  Q1={qs[2]:.3f}  '
      f'med={qs[3]:.3f}  Q3={qs[4]:.3f}  95%={qs[5]:.3f}  max={qs[6]:.3f}')
print(f'  ref target  mean=0.800 std=0.029 (reach with enough WWW iters)')

# 2) Spatial-coverage check - tile the canonical box into 1 µm cells and
# count cells with no vertex. Reference example: 54.5% empty (Poisson at
# density 0.74 verts/µm³ would naturally give ~44% empty). The thing the
# old configuration-model seed got wrong was *clusters* of empty cells -
# multi-µm voids. The Poisson-disk seed used now should give a roughly
# Poisson-like empty-cell pattern with no large connected void region.
half = BOX / 2.0
verts = np.vstack([p1, p2])
verts_canon = verts - BOX * np.round(verts / BOX)
n_cells = int(np.ceil(BOX))
edges_grid = np.linspace(-half, half, n_cells + 1)
H, _ = np.histogramdd(verts_canon, bins=(edges_grid, edges_grid, edges_grid))
empty = int(np.sum(H == 0))
print(f'1 µm³ vertex coverage: {H.size} cells, {empty} empty '
      f'({empty / H.size:.1%})  (reference: 54.5%)')

try:
    from scipy.ndimage import label
    labeled, n_components = label(H == 0)
    sizes = sorted((int((labeled == c).sum()) for c in range(1, n_components + 1)),
                   reverse=True)
    print(f'  largest empty clusters: {sizes[:5]}  '
          f'(big single cluster is normal at this density due to PBC '
          f'percolation; what was *wrong* before was a big cluster on a '
          f'box face)')
except ImportError:
    pass

print(f'box span   : x [{p1[:,0].min():.3f}, {p1[:,0].max():.3f}] '
      f'y [{p1[:,1].min():.3f}, {p1[:,1].max():.3f}] '
      f'z [{p1[:,2].min():.3f}, {p1[:,2].max():.3f}]')

# 3) Voxel-density uniformity - the test that surfaced the void-clustering
# issue. Bin rod midpoints into a 4x4x4 grid (cells of side ~2.86 µm at
# BOX=11.44) and measure spread + boundary-vs-interior bias. The 1µm
# coverage check above is too fine to see this - at 1µm there are ~3
# midpoints/cell, so empty cells dominate either way. The 4³ grid has
# ~26 midpoints expected per cell.
#
# Reference example (lsu_example_ends.txt, BOX=11.44):
#    4³ voxels:  std=3.65  min=17  max=35   corner sum=221  centre sum=183
#                                           corner/centre ratio=1.21
# Pre-fix run (lsu_generated_4.txt, BOX=11.44, relax_global_every=1000):
#    4³ voxels:  std=9.72  min=7   max=55   corner sum=222  centre sum=233
#                                           corner/centre ratio=0.95 (flipped)
# Pass criteria for the gated-fallback fix: std <= 4.0 AND
# corner/centre ratio in [1.0, 1.4].
midpts = (p1 + p2) / 2.0
midpts_canon = midpts - BOX * np.round(midpts / BOX)
vbins = np.linspace(-half, half, 5)
Hv, _ = np.histogramdd(midpts_canon, bins=(vbins, vbins, vbins))
# Boundary mask: any voxel with at least one index in {0, 3} along any axis.
bdry_mask = np.zeros(Hv.shape, dtype=bool)
bdry_mask[0, :, :] = bdry_mask[-1, :, :] = True
bdry_mask[:, 0, :] = bdry_mask[:, -1, :] = True
bdry_mask[:, :, 0] = bdry_mask[:, :, -1] = True
# Corner mask: 8 voxels with indices all in {0, 3}.
corner_mask = np.zeros(Hv.shape, dtype=bool)
for ix in (0, -1):
    for iy in (0, -1):
        for iz in (0, -1):
            corner_mask[ix, iy, iz] = True
# Centre mask: 8 voxels with indices all in {1, 2} (the inner 2x2x2).
centre_mask = np.zeros(Hv.shape, dtype=bool)
centre_mask[1:3, 1:3, 1:3] = True
expected = midpts_canon.shape[0] / Hv.size
print(f'4³ voxel midpoints (cell ~{BOX/4:.2f} µm, expected {expected:.1f}/cell):')
print(f'  spread     mean={Hv.mean():.2f} std={Hv.std():.2f} '
      f'min={Hv.min():.0f} max={Hv.max():.0f}')
print(f'  bdry/int   bdry mean={Hv[bdry_mask].mean():.2f}  '
      f'int mean={Hv[~bdry_mask].mean():.2f}')
print(f'  corner/centre  corner sum={Hv[corner_mask].sum():.0f}  '
      f'centre sum={Hv[centre_mask].sum():.0f}  '
      f'ratio={Hv[corner_mask].sum() / max(Hv[centre_mask].sum(), 1):.2f}')
print(f'  ref target std=3.65 corner/centre=1.21  '
      f'(pass: std<=4.0 AND ratio in [1.0, 1.4])')

In [ ]:
import tools

srs_rods = tools.srs_crystal_rods(num_vertices=1000, box=11.44, d0=0.8)

results = []
for source, lbl in [
    (srs_rods,                                  'srs (crystal)'),
    ('Example/lsu_example_ends.txt',            'reference'),
    (filename,                                  'This Run'),
]:
    results.append(tools.analyze_network(
        source, box=11.44, d0=0.8, label=lbl, verbose=False,
        k_modes_max_2d=16,
    ))

tools._print_comparison(results)

fig, axes = tools.plot_comparison(
    results,
    ring_range=(4, 12),
    sk_logy=True,
    sk_2d_log=True,
    sk_2d_shared_clim=True,
    sk_2d_interp='bilinear',
)
# fig.savefig('Example/lsu_compare_with_srs.png', dpi=150, bbox_inches='tight')


In [ ]:
# shutdown(delay_minutes=2)